### 🏆 최종 실습 과제: 실제 도서 리뷰 데이터로 숨은 목소리 찾기

지금까지 배운 토픽 모델링과 군집화 기법을 실제 데이터에 적용하여 독자들이 남긴 리뷰 속에 숨겨진 다양한 목소리와 주제를 발견해 봅시다.

**과제 목표:**
교보문고의 특정 베스트셀러 도서에 대한 리뷰를 직접 수집(크롤링)하고, 비지도 학습을 통해 리뷰들을 주제별로 묶고(토픽 모델링), 비슷한 내용의 리뷰들을 그룹화(군집화)하여 인사이트를 도출합니다.

#### 📚 1단계: 데이터 수집 (웹 크롤링)

먼저 분석할 리뷰 데이터를 수집해야 합니다. 제가 알려드리는 방식으로 수집을 해보세요(별도 교육)

* **대상 도서:** 트렌드 코리아 2025
* **대상 URL:** `https://product.kyobobook.co.kr/detail/S000214208202`
* **수집 내용:** 리뷰 텍스트

In [1]:
"""
fetch("https://product.kyobobook.co.kr/api/review/list?page=1&pageLimit=10&reviewSort=001&revwPatrCode=000&saleCmdtid=S000214208202", {
  "headers": {
    "accept": "*/*",
    "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "priority": "u=1, i",
    "sec-ch-ua": "\"Google Chrome\";v=\"137\", \"Chromium\";v=\"137\", \"Not/A)Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin"
  },
  "referrer": "https://product.kyobobook.co.kr/detail/S000214208202",
  "referrerPolicy": "strict-origin-when-cross-origin",
  "body": null,
  "method": "GET",
  "mode": "cors",
  "credentials": "include"
});
"""

# 교보문고 API를 통한 리뷰 데이터 수집
import requests
import pandas as pd
import json
import time

# 리뷰 데이터를 저장할 리스트
all_reviews = []

# 여러 페이지에서 리뷰 수집 (1~10페이지)
for page in range(1, 21):
    url = f"https://product.kyobobook.co.kr/api/review/list?page=1&pageLimit=10&reviewSort=001&revwPatrCode=000&saleCmdtid=S000214208202"
    
    headers = {
        "accept": "*/*",
        "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "cache-control": "no-cache",
        "pragma": "no-cache",
        "priority": "u=1, i",
        "sec-ch-ua": "\"Google Chrome\";v=\"137\", \"Chromium\";v=\"137\", \"Not/A)Brand\";v=\"24\"",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "\"macOS\"",
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            
            # 리뷰 데이터 추출 - API 응답 구조에 맞게 수정
            if 'data' in data and 'reviewList' in data['data']:
                reviews = data['data']['reviewList']
                for review in reviews:
                    review_text = review.get('revwCntt', '').strip()
                    if review_text:  # 빈 리뷰가 아닌 경우만 추가
                        all_reviews.append({
                            'review_text': review_text,
                            'rating': review.get('revwRvgr', 0),
                            'date': review.get('cretDttm', '')
                        })
            
            print(f"페이지 {page} 수집 완료 - 현재까지 {len(all_reviews)}개 리뷰")
            
        else:
            print(f"페이지 {page} 요청 실패: {response.status_code}")
            
    except Exception as e:
        print(f"페이지 {page} 수집 중 오류 발생: {e}")
    
    # 서버 부하 방지를 위한 대기
    time.sleep(1)

# DataFrame으로 변환
review_df = pd.DataFrame(all_reviews)
print(f"\n총 {len(review_df)}개의 리뷰를 수집했습니다.")
print(review_df.head())

# 빈 리뷰 제거cc
review_df = review_df[review_df['review_text'].str.strip() != '']
print(f"\n전처리 후 {len(review_df)}개의 리뷰가 남았습니다.")

페이지 1 수집 완료 - 현재까지 10개 리뷰
페이지 2 수집 완료 - 현재까지 20개 리뷰
페이지 3 수집 완료 - 현재까지 30개 리뷰
페이지 4 수집 완료 - 현재까지 40개 리뷰
페이지 5 수집 완료 - 현재까지 50개 리뷰
페이지 6 수집 완료 - 현재까지 60개 리뷰
페이지 7 수집 완료 - 현재까지 70개 리뷰
페이지 8 수집 완료 - 현재까지 80개 리뷰
페이지 9 수집 완료 - 현재까지 90개 리뷰
페이지 10 수집 완료 - 현재까지 100개 리뷰
페이지 11 수집 완료 - 현재까지 110개 리뷰
페이지 12 수집 완료 - 현재까지 120개 리뷰
페이지 13 수집 완료 - 현재까지 130개 리뷰
페이지 14 수집 완료 - 현재까지 140개 리뷰
페이지 15 수집 완료 - 현재까지 150개 리뷰
페이지 16 수집 완료 - 현재까지 160개 리뷰
페이지 17 수집 완료 - 현재까지 170개 리뷰
페이지 18 수집 완료 - 현재까지 180개 리뷰
페이지 19 수집 완료 - 현재까지 190개 리뷰
페이지 20 수집 완료 - 현재까지 200개 리뷰

총 200개의 리뷰를 수집했습니다.
                                         review_text  rating  \
0                          이 책이 나오면 겨울이 온다라고 생각이 듭니다       4   
1  매년 나오는데  24년트렌드는 구매하고 읽지도 못하고 한해가 거의 지나가네요 \n2...       3   
2           10분만 투자하면 전체 내용을 파악할 수 있는, 가벼이 훑어보면 되는 책       2   
3                                  매년 운세 보듯이 보게 되는 책       4   
4  소비자 트렌드 관점에서는 새로운 이슈 출현을 강조 하고자 했으나 점점 트렌드를 억측...       2   

                         date  
0  2024-09-05 09:39:


#### 🔍 2단계: 토픽 모델링 (LDA)으로 리뷰 주제 파악하기

수집한 리뷰들에는 어떤 숨겨진 주제들이 있을지 LDA를 통해 분석해 봅시다.

1.  **DTM 생성:** 전처리된 'processed' 데이터를 `CountVectorizer`를 사용하여 DTM(단어-문서 행렬)으로 변환하세요.
2.  **LDA 모델 학습:** `LatentDirichletAllocation`을 사용해 **4개의 토픽**을 추출해 보세요.
3.  **결과 해석:**
    * 각 토픽을 대표하는 상위 5~7개의 키워드를 출력하세요.
    * 키워드를 바탕으로 각 토픽에 **이름을 붙여보세요.** 예를 들어, "실천과 변화", "선물 및 추천", "번역 및 가독성" 등과 같이 해석할 수 있습니다. 이를 통해 독자들이 어떤 관점에서 이 책을 평가하는지 파악할 수 있습니다.

In [5]:
import re
from kiwipiepy import Kiwi

# 한국어 텍스트 전처리 (Kiwi 사용)
kiwi = Kiwi()
kiwi.add_user_word('트렌드', 'NNG')
kiwi.add_user_word('코리아', 'NNG')

def preprocess_korean_text(text):
    # 특수문자 제거
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', str(text))
    
    # 형태소 분석하여 명사만 추출
    tokens = kiwi.tokenize(text)
    nouns = [token.form for token in tokens 
             if token.tag in ['NNG', 'NNP'] and len(token.form) > 1]
    
    return ' '.join(nouns)

# 텍스트 전처리 적용
review_df['processed'] = review_df['review_text'].apply(preprocess_korean_text)

# 빈 결과 제거
review_df = review_df[review_df['processed'].str.strip() != '']
print(f"전처리 완료: {len(review_df)}개 리뷰")

print("\n[전처리 결과 샘플]")
for i in range(min(3, len(review_df))):
    print(f"원본: {review_df.iloc[i]['review_text']}")
    print(f"전처리: {review_df.iloc[i]['processed']}")
    print()

전처리 완료: 200개 리뷰

[전처리 결과 샘플]
원본: 이 책이 나오면 겨울이 온다라고 생각이 듭니다
전처리: 겨울 생각

원본: 매년 나오는데  24년트렌드는 구매하고 읽지도 못하고 한해가 거의 지나가네요 
25년도는 어떤트렌드일까 궁금함때문에 구매버튼을 또 ~~
다음은 안살것같아요
전처리: 트렌드 구매 트렌드 구매 버튼 다음

원본: 10분만 투자하면 전체 내용을 파악할 수 있는, 가벼이 훑어보면 되는 책
전처리: 투자 전체 내용 파악



In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# LDA 토픽 모델링
# DTM 생성
count_vectorizer = CountVectorizer(max_df=0.85, min_df=1, max_features=50)
dtm = count_vectorizer.fit_transform(review_df['processed'])
feature_names = count_vectorizer.get_feature_names_out()

print(f"DTM 형태: {dtm.shape}")
print(f"추출된 단어: {list(feature_names)}")

# LDA 모델 학습 (4개 토픽)
lda_model = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=100)
lda_model.fit(dtm)

# 토픽별 주요 단어 출력
def display_lda_topics(model, feature_names, n_top_words=5):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        topics.append({'topic_id': topic_idx + 1, 'keywords': top_features})
        
        print(f"\n=== 토픽 {topic_idx + 1} ===")
        print(f"주요 키워드: {', '.join(top_features)}")
        
    return topics

print("\nLDA 토픽 분석 결과:")
topics = display_lda_topics(lda_model, feature_names)

DTM 형태: (200, 50)
추출된 단어: ['강조', '겨울', '경제', '관점', '구매', '굿즈', '극대', '근거', '기대', '기쁨', '내용', '느낌', '다음', '데이터', '마무리', '매년', '모두', '방향', '시간', '시류', '심정', '억측', '열람', '영화', '예약', '올해', '운세', '응답', '이슈', '이용', '일부', '일상', '일터', '자료', '자원', '전체', '정체', '증가', '지속', '지향', '출현', '카드사', '콘텐츠', '태도', '투자', '트렌드', '트코', '파악', '행복', '현대']

LDA 토픽 분석 결과:

=== 토픽 1 ===
주요 키워드: 트렌드, 구매, 강조, 근거, 억측

=== 토픽 2 ===
주요 키워드: 태도, 굿즈, 시류, 겨울, 매년

=== 토픽 3 ===
주요 키워드: 트코, 올해, 느낌, 마무리, 겨울

=== 토픽 4 ===
주요 키워드: 방향, 시간, 일상, 현대, 자원



#### 🧩 3단계: K-Means 군집화로 유사 리뷰 그룹화하기

비슷한 내용을 담고 있는 리뷰들을 그룹으로 묶어 봅시다.

1.  **TF-IDF 행렬 생성:** 전처리된 'processed' 데이터를 `TfidfVectorizer`를 사용하여 TF-IDF 행렬로 변환하세요.
2.  **K-Means 모델 학습:** `KMeans`를 사용하여 **4개의 군집**으로 리뷰들을 나누세요.
3.  **결과 분석:**
    * 원본 `review_df`에 'cluster\_id' 컬럼을 추가하여 각 리뷰가 어떤 군집에 속하는지 확인하세요.
    * 각 군집별로 리뷰 내용을 몇 개씩 출력하여, 그룹이 어떤 기준으로 묶였는지(e.g., 긍정적 실천 후기, 책의 구성 칭찬, 배송 관련 등) 그 특징을 분석해 보세요.


In [7]:
# K-Means 군집화
# TF-IDF 벡터화
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=1, max_features=50)
tfidf_matrix = tfidf_vectorizer.fit_transform(review_df['processed'])

print(f"TF-IDF 행렬 형태: {tfidf_matrix.shape}")

# K-Means 군집화 (4개 군집)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(tfidf_matrix)

# 군집 결과를 데이터프레임에 추가
review_df['cluster_id'] = cluster_labels

print("\n군집화 결과:")
cluster_counts = review_df['cluster_id'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = count / len(review_df) * 100
    print(f"군집 {cluster_id}: {count}개 리뷰 ({percentage:.1f}%)")

# 각 군집별 대표 리뷰 출력
print("\n각 군집별 대표 리뷰:")
for cluster_id in sorted(review_df['cluster_id'].unique()):
    cluster_reviews = review_df[review_df['cluster_id'] == cluster_id]
    print(f"\n=== 군집 {cluster_id} ({len(cluster_reviews)}개 리뷰) ===")
    
    # 각 군집에서 2개의 대표 리뷰 선택
    sample_reviews = cluster_reviews.head(2)
    for idx, review in enumerate(sample_reviews.itertuples(), 1):
        print(f"{idx}. {review.review_text}")
        print(f"   평점: {review.rating}점, 처리된 텍스트: {review.processed}\n")

TF-IDF 행렬 형태: (200, 50)

군집화 결과:
군집 0: 20개 리뷰 (10.0%)
군집 1: 40개 리뷰 (20.0%)
군집 2: 20개 리뷰 (10.0%)
군집 3: 120개 리뷰 (60.0%)

각 군집별 대표 리뷰:

=== 군집 0 (20개 리뷰) ===
1. '현대사회에서 가장 큰 자원은 시간이다.' 라는 응답이 년도의 증가와 함께 늘어가고 있다. 콘텐츠의 열람도 배속으로, 영화도 숏폼으로, 예약도 앱을 이용하여 기다리는 시간을 줄이는 방향으로...
모두가 시간의 효율을 극대화하려 한다.
일상에서도, 일터에서도, 쇼핑에서도...

2025년의 트렌드도 그런 방향으로의 변화의 지속일 것이다. 세계경제의 발전이 더뎌지고, 정체되면서 우리의 지향도 작은 것을 아끼고, 소소한 일상에서 기쁨과 행복을 느끼는 방향으로...
   평점: 4점, 처리된 텍스트: 현대 사회 자원 시간 응답 증가 콘텐츠 열람 영화 예약 이용 시간 방향 모두 시간 효율 극대 일상 일터 쇼핑 트렌드 방향 변화 지속 세계 경제 발전 정체 지향 일상 기쁨 행복 방향

2. '현대사회에서 가장 큰 자원은 시간이다.' 라는 응답이 년도의 증가와 함께 늘어가고 있다. 콘텐츠의 열람도 배속으로, 영화도 숏폼으로, 예약도 앱을 이용하여 기다리는 시간을 줄이는 방향으로...
모두가 시간의 효율을 극대화하려 한다.
일상에서도, 일터에서도, 쇼핑에서도...

2025년의 트렌드도 그런 방향으로의 변화의 지속일 것이다. 세계경제의 발전이 더뎌지고, 정체되면서 우리의 지향도 작은 것을 아끼고, 소소한 일상에서 기쁨과 행복을 느끼는 방향으로...
   평점: 4점, 처리된 텍스트: 현대 사회 자원 시간 응답 증가 콘텐츠 열람 영화 예약 이용 시간 방향 모두 시간 효율 극대 일상 일터 쇼핑 트렌드 방향 변화 지속 세계 경제 발전 정체 지향 일상 기쁨 행복 방향


=== 군집 1 (40개 리뷰) ===
1. 매년 나오는데  24년트렌드는 구매하고 읽지도 못하고 한해가 거의 지나가네요 
25년도는 어

#### 📊 4단계: 시각화로 군집 결과 확인하기

군집화 결과를 PCA나 t-SNE를 이용해 2차원 공간에 시각화하여 그룹이 잘 형성되었는지 확인합니다.

1.  **차원 축소:** `PCA`나 `t-SNE`를 사용해 TF-IDF 행렬을 2개의 주성분으로 축소하세요.
2.  **산점도 시각화:** `plotly.express`를 사용해 결과를 산점도로 그리세요.
    * 각 점의 색상은 `cluster_id`로 구분합니다.
    * 마우스를 점 위에 올렸을 때 원본 리뷰(`review`)가 표시되도록 설정하여, 각 군집의 분포와 특징을 시각적으로 탐색해 보세요.

In [8]:
import plotly.express as px
import plotly.graph_objects as go
import pyLDAvis
import pyLDAvis.lda_model

# 차원 축소 및 시각화
# t-SNE 적용 (데이터가 적은 경우 perplexity 조정)
perplexity_value = min(30, len(review_df) - 1) if len(review_df) > 1 else 1
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity_value)
tsne_components = tsne.fit_transform(tfidf_matrix.toarray())

# 시각화용 데이터프레임 생성
viz_df = review_df.copy()
viz_df['tsne_x'] = tsne_components[:, 0]
viz_df['tsne_y'] = tsne_components[:, 1]
viz_df['cluster_label'] = 'Cluster ' + viz_df['cluster_id'].astype(str)

# t-SNE 시각화
fig1 = px.scatter(
    viz_df,
    x='tsne_x',
    y='tsne_y',
    color='cluster_label',
    hover_data=['review_text', 'rating'],
    title='K-Means 군집화 결과 (t-SNE 2D 시각화)',
    labels={
        'tsne_x': 't-SNE Component 1',
        'tsne_y': 't-SNE Component 2',
        'cluster_label': '군집'
    },
    width=800,
    height=600
)

fig1.update_traces(marker=dict(size=12, opacity=0.8))
fig1.show()

print("t-SNE 시각화: 같은 색상의 점들이 가까이 모여있으면 군집화가 잘 된 것입니다.")


t-SNE 시각화: 같은 색상의 점들이 가까이 모여있으면 군집화가 잘 된 것입니다.


In [9]:
# PCA 시각화
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(tfidf_matrix.toarray())

viz_df['pca_x'] = pca_components[:, 0]
viz_df['pca_y'] = pca_components[:, 1]

fig2 = px.scatter(
    viz_df,
    x='pca_x',
    y='pca_y',
    color='cluster_label',
    hover_data=['review_text', 'rating'],
    title='K-Means 군집화 결과 (PCA 2D 시각화)',
    labels={
        'pca_x': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} 설명력)',
        'pca_y': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} 설명력)',
        'cluster_label': '군집'
    },
    width=800,
    height=600
)

fig2.update_traces(marker=dict(size=12, opacity=0.8))
fig2.show()

print(f"PCA 설명력: PC1 {pca.explained_variance_ratio_[0]:.1%}, PC2 {pca.explained_variance_ratio_[1]:.1%}")
print(f"총 설명력: {pca.explained_variance_ratio_.sum():.1%}")

# pyLDAvis 토픽 시각화 시도
try:
    pyLDAvis.enable_notebook()
    vis_data = pyLDAvis.lda_model.prepare(lda_model, dtm, count_vectorizer)
    pyLDAvis.display(vis_data)
except Exception as e:
    print(f"pyLDAvis 시각화 오류: {e}")
    print("대신 토픽 분포를 텍스트로 출력합니다.")
    
    # 각 문서의 주요 토픽 계산
    doc_topic_dist = lda_model.transform(dtm)
    dominant_topics = doc_topic_dist.argmax(axis=1)
    
    topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
    print("\n각 토픽별 문서 분포:")
    for topic_id, count in topic_counts.items():
        percentage = count / len(review_df) * 100
        print(f"토픽 {topic_id + 1}: {count}개 문서 ({percentage:.1f}%)")

PCA 설명력: PC1 13.0%, PC2 11.2%
총 설명력: 24.2%
